In [ ]:
# =========================
# Cell 1 — imports + style/helpers
# =========================
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("retina")

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter

# -----------------------------
# Style controls
# -----------------------------
BASE_FONTSIZE      = 20
LABEL_FONTSIZE     = 24
TITLE_FONTSIZE     = 22
TICK_FONTSIZE      = 16
COLORBAR_FONTSIZE  = 18

LEGEND_VPAD = 0.18

BASE_CMAP_NAME = "viridis"
CVAL_MIN   = 0.08
CVAL_MAX   = 0.92
CVAL_GAMMA = 0.95

STYLE_SOLID = "-"
DASH_SEQ    = (6, 3)

COLORBAR_WIDTH  = 0.62
COLORBAR_HEIGHT = 0.75

TAIL_FIT_FRAC = 0.25   # used to fit one multiplicative constant for each dashed asymptotic curve


def set_pub_style():
    mpl.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,

        "font.family": "serif",
        "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",

        "font.size": BASE_FONTSIZE,
        "axes.labelsize": LABEL_FONTSIZE,
        "axes.titlesize": TITLE_FONTSIZE,
        "xtick.labelsize": TICK_FONTSIZE,
        "ytick.labelsize": TICK_FONTSIZE,
        "legend.fontsize": TICK_FONTSIZE,

        "axes.linewidth": 0.9,
        "lines.linewidth": 1.8,
        "lines.solid_capstyle": "round",
        "lines.solid_joinstyle": "round",

        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.minor.size": 2.5,
        "ytick.minor.size": 2.5,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,

        "axes.grid": False,
    })


set_pub_style()


def shade_from_index(i, n, light=CVAL_MIN, dark=CVAL_MAX, gamma=CVAL_GAMMA):
    if n <= 1:
        return 0.5 * (light + dark)
    t = i / (n - 1)
    t = t ** gamma
    return light + (dark - light) * t


def prettify_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(which="both", top=False, right=False)
    ax.grid(True, which="major", linewidth=0.6, alpha=0.22)
    ax.grid(False, which="minor")


def prettify_log_axis(ax, axis="y"):
    locator_major = LogLocator(base=10.0, numticks=10)
    locator_minor = LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100)
    formatter = LogFormatterMathtext(base=10.0)
    if axis == "y":
        ax.yaxis.set_major_locator(locator_major)
        ax.yaxis.set_minor_locator(locator_minor)
        ax.yaxis.set_major_formatter(formatter)
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.xaxis.set_major_locator(locator_major)
        ax.xaxis.set_minor_locator(locator_minor)
        ax.xaxis.set_major_formatter(formatter)
        ax.xaxis.set_minor_formatter(NullFormatter())


def add_alpha_colorbar_horizontal_single(
    cax, alphas, colors, label=r"$\alpha$", cb_fs=COLORBAR_FONTSIZE
):
    alphas = np.asarray(alphas)
    n = len(alphas)

    img = np.zeros((1, n, 4))
    for i in range(n):
        img[0, i, :] = colors[i]

    da = float(np.mean(np.diff(alphas))) if n > 1 else 1.0
    x0, x1 = float(alphas[0] - da / 2), float(alphas[-1] + da / 2)

    cax.imshow(img, origin="lower", aspect="auto", extent=(x0, x1, 0, 1))
    cax.set_ylim(0, 1)
    cax.set_yticks([])

    cax.set_xticks(alphas)
    cax.tick_params(
        axis="x", which="both",
        bottom=True, labelbottom=True,
        top=False, labeltop=False,
        length=0, width=0, pad=3,
        labelsize=cb_fs
    )

    cax.set_frame_on(False)
    for sp in cax.spines.values():
        sp.set_visible(False)

    cax.set_xlabel(label, fontsize=cb_fs)
    cax.xaxis.set_label_position("top")
    return cax


def center_shrink_axis(ax, width_scale=COLORBAR_WIDTH, height_scale=COLORBAR_HEIGHT):
    pos = ax.get_position()
    new_w = pos.width * width_scale
    new_h = pos.height * height_scale
    new_x = pos.x0 + (pos.width - new_w) / 2
    new_y = pos.y0 + (pos.height - new_h) / 2
    ax.set_position([new_x, new_y, new_w, new_h])

In [ ]:
# =========================
# Cell 2 — kernels, recursions, p_* solver, asymptotic helpers
# =========================
pi = np.pi


def kappa_relu(rho):
    rho = np.clip(rho, -1.0, 1.0)
    return (1.0 / (2.0 * pi)) * (
        np.sqrt(np.maximum(0.0, 1.0 - rho**2)) + rho * (pi - np.arccos(rho))
    )


def tilde_q_erf(q, alpha):
    x = (2.0 * alpha**2 * q) / (1.0 + 2.0 * alpha**2 * q)
    x = np.clip(x, -1.0, 1.0)
    return (2.0 / pi) * np.arcsin(x)


def tilde_p_erf(q, p, alpha):
    x = (2.0 * alpha**2 * p) / (1.0 + 2.0 * alpha**2 * q)
    x = np.clip(x, -1.0, 1.0)
    return (2.0 / pi) * np.arcsin(x)


def simulate_preln_full_chi(
    num_layers,
    p0,
    n_tokens,
    sigma_W1,
    sigma_W2,
    sigma_O,
    sigma_V,
    sigma_A,
    q0=1.0,
):
    """
    pre-LN: use the full chi product chi_att * chi_mlp
    and define J^{l,0} = prod_{k=0}^{l-1} chi[k], J^{0,0}=1
    """
    L = int(num_layers)
    n = float(n_tokens)
    eps = 1e-12

    q = np.zeros(L + 1, dtype=float)
    p = np.zeros(L + 1, dtype=float)
    q[0], p[0] = float(q0), float(p0)

    chi_att = np.zeros(L, dtype=float)
    chi_mlp = np.zeros(L, dtype=float)

    att_scale = (sigma_O**2) * (sigma_V**2)
    mlp_scale = (sigma_W1**2) * (sigma_W2**2)

    for l in range(L):
        ql, pl = q[l], p[l]

        rho = float(np.clip(pl / (ql + eps), -1.0, 1.0))
        beta = np.exp((sigma_A**2) * (rho - 1.0))
        gamma = np.exp((sigma_A**2) * rho * (rho - 1.0))
        denom = 1.0 + (n - 1.0) * beta

        chi_att[l] = 1.0

        qh = ql + att_scale * rho
        ph = pl + att_scale * rho

        chi_mlp[l] = 1.0 + mlp_scale / (2.0 * (qh + eps))

        rho_half = float(np.clip(ph / (qh + eps), -1.0, 1.0))
        dq_mlp = 0.5 * mlp_scale
        dp_mlp = mlp_scale * kappa_relu(rho_half)

        q[l + 1] = qh + dq_mlp
        p[l + 1] = ph + dp_mlp

    chi = chi_att * chi_mlp
    return chi, q, p


def simulate_derf_chi_mlp(
    num_layers,
    p0,
    n_tokens,
    alpha,
    sigma_W1,
    sigma_W2,
    sigma_O,
    sigma_V,
    sigma_A,
    q0=1.0,
):
    """
    derf: match the asymptotic-exponents notebook convention
    chi_attn[l] = 1, so chi[l] = chi_mlp[l]
    """
    L = int(num_layers)
    n = float(n_tokens)
    eps = 1e-12

    q = np.zeros(L + 1, dtype=float)
    p = np.zeros(L + 1, dtype=float)
    q[0], p[0] = float(q0), float(p0)

    chi_mlp = np.zeros(L, dtype=float)

    att_scale = (sigma_O**2) * (sigma_V**2)
    mlp_scale = (sigma_W1**2) * (sigma_W2**2)

    for l in range(L):
        ql, pl = q[l], p[l]

        uq = tilde_q_erf(ql, alpha)
        up = tilde_p_erf(ql, pl, alpha)

        qh = ql + att_scale * up
        ph = pl + att_scale * up

        chi_mlp[l] = 1.0 + mlp_scale * (2.0 * alpha**2 / pi) * (
            1.0 / np.sqrt(1.0 + 4.0 * alpha**2 * qh)
        )

        u_half = tilde_q_erf(qh, alpha)
        v_half = tilde_p_erf(qh, ph, alpha)
        rho_z = np.clip(v_half / (u_half + eps), -1.0, 1.0)

        dq_mlp = 0.5 * mlp_scale * u_half
        dp_mlp = mlp_scale * u_half * kappa_relu(rho_z)

        q[l + 1] = qh + dq_mlp
        p[l + 1] = ph + dp_mlp

    return chi_mlp, q, p


def p_of_c(c):
    c = np.clip(c, 0.0, 1.0)
    return (2.0 / pi) * np.arcsin(c)
    # return c


def kappa(x):
    x = np.clip(x, -1.0, 1.0)
    return (1.0 / (2.0 * pi)) * (
        np.sqrt(np.maximum(0.0, 1.0 - x**2)) + x * (pi - np.arccos(x))
    )


def bisect_root(fun, a, b, tol=1e-12, max_iter=200):
    fa, fb = fun(a), fun(b)
    if np.isnan(fa) or np.isnan(fb):
        raise ValueError("NaN encountered at bracket endpoints.")
    if fa == 0.0:
        return a
    if fb == 0.0:
        return b
    if fa * fb > 0:
        raise ValueError("Bisection requires a sign change on [a, b].")

    lo, hi = a, b
    flo, fhi = fa, fb
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = fun(mid)
        if abs(fmid) <= tol or (hi - lo) <= tol:
            return mid
        if flo * fmid <= 0:
            hi, fhi = mid, fmid
        else:
            lo, flo = mid, fmid
    return 0.5 * (lo + hi)


def find_p_star(sigma_OV, sigma_21, grid_n=4001):
    """
    Same numerical selection as your asymptotic notebook:
    choose the largest root with c < 1, then set p_* = (2/pi) arcsin(c_*).
    """
    def g(c):
        pc = p_of_c(c)
        num = (sigma_OV**2) * pc + (sigma_21**2) * kappa(pc)
        den = (sigma_OV**2) * pc + (sigma_21**2) / 2.0
        return num / den

    def f(c):
        return g(c) - c

    xs = np.linspace(0.0, 1.0, grid_n)
    fs = np.array([f(x) for x in xs])

    roots = []

    near = np.where(np.abs(fs) < 1e-8)[0]
    for idx in near:
        roots.append(xs[idx])

    for i in range(grid_n - 1):
        a, b = xs[i], xs[i + 1]
        fa, fb = fs[i], fs[i + 1]
        if fa == 0.0:
            roots.append(a)
        if fa * fb < 0:
            roots.append(bisect_root(f, a, b, tol=1e-12, max_iter=200))

    roots = np.array(sorted(roots))
    dedup = []
    for r in roots:
        if not dedup or abs(r - dedup[-1]) > 1e-7:
            dedup.append(r)
    roots = np.array(dedup)

    if roots.size == 0:
        best = xs[np.argmin(np.abs(fs))]
        roots = np.array([best])

    cand = roots[roots < (1.0 - 1e-10)]
    if cand.size > 0:
        c_star = float(np.max(cand))
    else:
        c_star = float(xs[xs < 1.0][-1])

    p_star = float(p_of_c(c_star))
    return p_star, c_star, roots


def logJ_from_chi(chi):
    eps = 1e-300
    return np.concatenate([[0.0], np.cumsum(np.log(np.maximum(chi, eps)))])


def fit_log_amplitude(logJ_data, log_shape, fit_frac=TAIL_FIT_FRAC):
    idx = np.arange(logJ_data.size)
    start = max(1, int(np.floor((1.0 - fit_frac) * (logJ_data.size - 1))))
    mask = (idx >= start) & np.isfinite(logJ_data) & np.isfinite(log_shape)
    if not np.any(mask):
        return 0.0
    return float(np.mean(logJ_data[mask] - log_shape[mask]))


def safe_exp(x):
    return np.exp(np.clip(x, -700.0, 700.0))

In [ ]:
# =========================
# Cell 3 — parameters + compute numerical APJN and asymptotics
# =========================

import math

# ---- parameters ----
# This alpha grid matches the asymptotic-exponents notebook.
L        = 1000
n_tokens = 197
p0       = 0.03
q0       = 1.0

# sigma_W1 = 0.64
# sigma_W2 = 1.28
# sigma_O  = 0.64
# sigma_V  = 0.64
# sigma_A  = 0.64 * 0.64

sigma_W1=0.64 * math.sqrt(768 / 1024) * 2
sigma_W2=1.28 * math.sqrt(768 / 1024) * 2
sigma_O=0.64 * math.sqrt(768 / 1024) * 2
sigma_V=0.64 * math.sqrt(768 / 1024) * 2
sigma_A=0.64 * 0.64 * 768 / 1024

alphas   = np.arange(0.1, 1.9 + 1e-9, 0.2)

# fit constants using only the last fraction of points
FIT_FRACTION = 0.30
# --------------------

sigma_OV = sigma_O * sigma_V
sigma_21 = sigma_W1 * sigma_W2

zeta = ((sigma_21**2) / 2.0) / (((sigma_21**2) / 2.0) + sigma_OV**2)

p_star, c_star, roots = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)

print(f"sigma_OV = {sigma_OV:.8g}")
print(f"sigma_21 = {sigma_21:.8g}")
print(f"zeta     = {zeta:.8g}")
print(f"roots(c) = {roots}")
print(f"c_*      = {c_star:.15g}")
print(f"p_*      = {p_star:.15g}")

# colors
nA = len(alphas)
base_cmap = plt.get_cmap(BASE_CMAP_NAME)
cvals = [shade_from_index(i, nA) for i in range(nA)]
colors = [base_cmap(cv) for cv in cvals]

# l arrays
l_arr = np.arange(L + 1, dtype=float)
l_pos = np.arange(1, L + 1, dtype=float)

def get_tail_slice(n, fraction=0.30):
    """
    Return a slice selecting the last `fraction` of an array of length n.
    """
    n_tail = max(1, int(np.ceil(fraction * n)))
    return slice(n - n_tail, n)

def fit_additive_constant_log_tail(logJ, slope, x, fraction=0.40):
    """
    Fit C in logJ = slope * x + C using only the last `fraction` of points.
    Here logJ should be the full array including l=0, while x corresponds to l=1..L.
    """
    resid = np.asarray(logJ, dtype=float)[1:] - slope * np.asarray(x, dtype=float)
    tail = get_tail_slice(len(resid), fraction=fraction)
    y = resid[tail]
    mask = np.isfinite(y)
    if not np.any(mask):
        return 0.0
    return float(np.mean(y[mask]))

def fit_additive_constant_tail(y_data, y_shape, fraction=0.30):
    """
    Fit C in y_data = y_shape + C using only the last `fraction` of points.
    Both arrays should correspond to l=1..L.
    """
    resid = np.asarray(y_data, dtype=float) - np.asarray(y_shape, dtype=float)
    tail = get_tail_slice(len(resid), fraction=fraction)
    y = resid[tail]
    mask = np.isfinite(y)
    if not np.any(mask):
        return 0.0
    return float(np.mean(y[mask]))

tail = get_tail_slice(len(l_pos), FIT_FRACTION)
print(f"Fitting constants using l = {int(l_pos[tail][0])}, ..., {int(l_pos[tail][-1])} "
      f"(last {100*FIT_FRACTION:.0f}% of points)")

# ---- pre-LN numerical APJN ----
chi_ln, q_ln, p_ln = simulate_preln_full_chi(
    num_layers=L,
    p0=p0,
    n_tokens=n_tokens,
    sigma_W1=sigma_W1,
    sigma_W2=sigma_W2,
    sigma_O=sigma_O,
    sigma_V=sigma_V,
    sigma_A=sigma_A,
    q0=q0,
)

logJ_ln = logJ_from_chi(chi_ln)
J_ln = safe_exp(logJ_ln)

# theory: log J_asym(l) = zeta * log l + C_ln_fit
C_ln_fit = fit_additive_constant_log_tail(
    logJ_ln,
    zeta,
    np.log(l_pos),
    fraction=FIT_FRACTION,
)

logJ_ln_asym = np.zeros(L + 1, dtype=float)
logJ_ln_asym[1:] = zeta * np.log(l_pos) + C_ln_fit
J_ln_asym = safe_exp(logJ_ln_asym)

# ---- derf numerical APJN + asymptotics ----
derf_curves = []

for i, a in enumerate(alphas):
    chi_d, q_d, p_d = simulate_derf_chi_mlp(
        num_layers=L,
        p0=p0,
        n_tokens=n_tokens,
        alpha=float(a),
        sigma_W1=sigma_W1,
        sigma_W2=sigma_W2,
        sigma_O=sigma_O,
        sigma_V=sigma_V,
        sigma_A=sigma_A,
        q0=q0,
    )

    logJ_d = logJ_from_chi(chi_d)
    J_d = safe_exp(logJ_d)

    C_pref = float(a) * 2.0 / np.pi
    lam = (((sigma_21**2) / 2.0) + sigma_OV**2 * p_star) / (C_pref**2 * sigma_21**4)

    # theory: log J_asym(l) = sqrt(l/lambda) - 1/(8 lambda) + C_d_fit
    log_shape_d = np.zeros(L + 1, dtype=float)
    log_shape_d[1:] = np.sqrt(l_pos / lam) - (1.0 / (8.0 * lam))

    C_d_fit = fit_additive_constant_tail(
        logJ_d[1:],
        log_shape_d[1:],
        fraction=FIT_FRACTION,
    )

    logJ_d_asym = np.zeros(L + 1, dtype=float)
    logJ_d_asym[1:] = log_shape_d[1:] + C_d_fit
    J_d_asym = safe_exp(logJ_d_asym)

    derf_curves.append({
        "alpha": float(a),
        "color": colors[i],
        "lambda": lam,
        "C_fit": C_d_fit,
        "logJ": logJ_d,
        "J": J_d,
        "logJ_asym": logJ_d_asym,
        "J_asym": J_d_asym,
        "q": q_d,
        "p": p_d,
    })

print(f"\npre-LN fitted constant: C_ln_fit = {C_ln_fit:.8g}")

print("\nFirst few lambdas and fitted constants:")
for d in derf_curves[:4]:
    print(f"alpha={d['alpha']:.2f}, lambda={d['lambda']:.8g}, C_fit={d['C_fit']:.8g}")

In [ ]:
# =========================
# Cell 4 — modified plots
#   1) log-log: pre-LN only
#   2) (log J)^2 vs l: derf only
#   Asymptotics: grey, on top, customizable dash pattern
# =========================

# -------------------------
# Controls for grey asymptotic lines
# -------------------------
ASYMPTOTIC_COLOR = "0.5"
ASYMPTOTIC_ZORDER = 10
SOLID_ZORDER = 3
ASYMPTOTIC_LINEWIDTH = 2.2

# Choose:
#   "dashed"
#   "dash-dot-dash"
ASYM_STYLE_MODE = "dashed"

# Main dash controls
ASYM_DASH_LEN = 3.0
ASYM_GAP_LEN  = 6.0

# Used only for "dash-dot-dash"
ASYM_DOT_LEN = 0.8   # small value + round capstyle makes it look like a point


def get_asym_linestyle(mode="dashed", dash_len=6.0, gap_len=3.0, dot_len=0.8):
    """
    Returns a matplotlib linestyle tuple.
    """
    if mode == "dash-dot-dash":
        # repeating pattern: dash, gap, dot, gap, dash, gap
        return (0, (dash_len, gap_len, dot_len, gap_len, dash_len, gap_len))
    else:
        # plain dashed
        return (0, (dash_len, gap_len))


ASYM_LINESTYLE = get_asym_linestyle(
    mode=ASYM_STYLE_MODE,
    dash_len=ASYM_DASH_LEN,
    gap_len=ASYM_GAP_LEN,
    dot_len=ASYM_DOT_LEN,
)

# -----------------------------------
# Plot 1: log-log plot, pre-LN only
# -----------------------------------
fig1 = plt.figure(figsize=(5.2, 5.2))
ax1 = fig1.add_subplot(111)

# solid numerical curve
ax1.plot(
    l_arr[1:], J_ln[1:],
    color="black",
    linewidth=2.6,
    linestyle=STYLE_SOLID,
    label="pre-LN",
    zorder=SOLID_ZORDER
)

# grey asymptotic curve on top
ax1.plot(
    l_arr[1:], J_ln_asym[1:],
    color=ASYMPTOTIC_COLOR,
    linewidth=ASYMPTOTIC_LINEWIDTH,
    linestyle=ASYM_LINESTYLE,
    dash_capstyle="round",
    label="pre-LN asymptotic",
    zorder=ASYMPTOTIC_ZORDER
)

ax1.set_xscale("log")
ax1.set_yscale("log")
prettify_log_axis(ax1, "x")
prettify_log_axis(ax1, "y")
ax1.set_xlabel(r"$l$")
ax1.set_ylabel(r"$\mathcal{J}^{\,l,0}$")
ax1.set_xlim(1, L)
ax1.set_title(r"pre-LN: log-log plot of $\mathcal{J}^{\,l,0}$")
prettify_axes(ax1)
ax1.legend(frameon=False, loc="best")

plt.show()


# -----------------------------------
# Plot 2: (log J)^2 vs l, derf only
# -----------------------------------
fig2 = plt.figure(figsize=(5.2, 5.2))
gs2 = fig2.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1.0, 0.16],
    hspace=0.42 + LEGEND_VPAD
)
ax2 = fig2.add_subplot(gs2[0, 0])
cax2 = fig2.add_subplot(gs2[1, 0])
center_shrink_axis(cax2)

# first draw solid numerical curves
for d in derf_curves:
    ax2.plot(
        l_arr, d["logJ"]**2,
        color=d["color"],
        linestyle=STYLE_SOLID,
        zorder=SOLID_ZORDER
    )

# then draw grey asymptotic curves on top
for d in derf_curves:
    ax2.plot(
        l_arr, d["logJ_asym"]**2,
        color=ASYMPTOTIC_COLOR,
        linewidth=ASYMPTOTIC_LINEWIDTH,
        linestyle=ASYM_LINESTYLE,
        dash_capstyle="round",
        zorder=ASYMPTOTIC_ZORDER
    )

ax2.set_xlabel(r"$l$")
ax2.set_ylabel(r"$\left(\log \mathcal{J}^{\,l,0}\right)^2$")
ax2.set_xlim(0, L)
ax2.set_title(r"derf: $\left(\log \mathcal{J}^{\,l,0}\right)^2$ vs. $l$")
prettify_axes(ax2)

add_alpha_colorbar_horizontal_single(cax2, alphas, colors, label=r"$\alpha$")

plt.show()

In [ ]:
# =========================
# Cell — derf only: J vs l
# (pre-LN omitted)
# =========================

fig = plt.figure(figsize=(5.2, 5.2))
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1.0, 0.16],
    hspace=0.42 + LEGEND_VPAD
)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[1, 0])
center_shrink_axis(cax)

# first draw solid numerical curves
for d in derf_curves:
    ax.plot(
        l_arr,
        np.exp(d["logJ"]),   # use d["J"] if you already stored J directly
        color=d["color"],
        linestyle=STYLE_SOLID,
        zorder=SOLID_ZORDER
    )

# then draw grey asymptotic curves on top
for d in derf_curves:
    ax.plot(
        l_arr,
        np.exp(d["logJ_asym"]),   # use d["J_asym"] if available
        color=ASYMPTOTIC_COLOR,
        linewidth=ASYMPTOTIC_LINEWIDTH,
        linestyle=ASYM_LINESTYLE,
        dash_capstyle="round",
        zorder=ASYMPTOTIC_ZORDER
    )

ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\mathcal{J}^{\,l,0}$")
ax.set_xlim(0, L)
ax.set_title(r"derf: $\mathcal{J}^{\,l,0}$ vs. $l$")
prettify_axes(ax)

add_alpha_colorbar_horizontal_single(cax, alphas, colors, label=r"$\alpha$")

plt.show()

In [ ]:
# =========================
# Cell — derf only: log J vs l
# (pre-LN omitted)
# =========================

fig = plt.figure(figsize=(5.2, 5.2))
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1.0, 0.16],
    hspace=0.42 + LEGEND_VPAD
)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[1, 0])
center_shrink_axis(cax)

# first draw solid numerical curves
for d in derf_curves:
    ax.plot(
        l_arr,
        d["logJ"],
        color=d["color"],
        linestyle=STYLE_SOLID,
        zorder=SOLID_ZORDER
    )

# then draw grey asymptotic curves on top
for d in derf_curves:
    ax.plot(
        l_arr,
        d["logJ_asym"],
        color=ASYMPTOTIC_COLOR,
        linewidth=ASYMPTOTIC_LINEWIDTH,
        linestyle=ASYM_LINESTYLE,
        dash_capstyle="round",
        zorder=ASYMPTOTIC_ZORDER
    )

ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\log \mathcal{J}^{\,l,0}$")
ax.set_xlim(0, L)
ax.set_title(r"derf: $\log \mathcal{J}^{\,l,0}$ vs. $l$")
prettify_axes(ax)

add_alpha_colorbar_horizontal_single(cax, alphas, colors, label=r"$\alpha$")

plt.show()

**Small alpha behavior**

In [ ]:
# L        = 100
# n_tokens = 197
# p0       = 0.25
# q0       = 0.5

# sigma_W1 = 0.64
# sigma_W2 = 1.28
# sigma_O  = 0.64
# sigma_V  = 0.64
# sigma_A  = 0.64 * 0.64

# alphas   = np.arange(0.2, 0.25 + 1e-9, 0.05)

In [ ]:
# =========================
# New Cell 1 — helpers for l_* and small-alpha comparison
# =========================

DERF_DASHED_STYLE  = (0, (5.0, 3.0))
SMALL_ALPHA_STYLE  = (0, (4.0, 3.0))
SMALL_ALPHA_COLOR  = "0.6"
DERF_LINEWIDTH     = 2.0
SMALL_ALPHA_LW     = 1.8
LSTAR_MARKER_SIZE  = 34


def compute_l_star(alpha, sigma_21, q0, p0):
    """
    l_* = ln((alpha^2 q^0)^(-1)) / ln(1 + (1/2) sigma_21^2 (4/pi) alpha^2)
    """
    growth = 1.0 + (0.5 * (sigma_21**2) + sigma_OV**2) * (4.0 / np.pi) * (alpha**2) + ((4.0 / np.pi) * (alpha**2))**2 * 0.5 * (sigma_21**2) * sigma_OV**2

    return np.log(1.0 / ((alpha**2) * p0)) / np.log(growth)


# def compute_l_star(alpha, sigma_21, q0):
#     """
#     l_* = ln((alpha^2 q^0)^(-1)) / ln(1 + (1/2) sigma_21^2 (4/pi) alpha^2)
#     """
#     growth = 1.0 + (0.5 * (sigma_21**2)) * (4.0 / np.pi) * (alpha**2)

#     return np.log(1.0 / ((alpha**2) * q0)) / np.log(growth)


def compute_J_small_alpha(alpha, sigma_21, l_values):
    """
    J_small_alpha(l) = (1 + (1/2) sigma_21^2 (4 alpha^2 / pi))^l
    """
    growth = 1.0 + 0.5 * (sigma_21**2) * (4.0 / np.pi) * (alpha**2)
    l_values = np.asarray(l_values, dtype=float)
    return safe_exp(l_values * np.log(growth))


def interp_positive_curve_logy(x, y, x_star, eps=1e-300):
    """
    Interpolate y(x_star) by linear interpolation in log y.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_star_clip = float(np.clip(x_star, x[0], x[-1]))
    logy = np.log(np.maximum(y, eps))
    return float(np.exp(np.interp(x_star_clip, x, logy)))


def split_curve_at_l_star(x, y, l_star):
    """
    Returns dashed part (l < l_*) and solid part (l > l_*),
    with the interpolated point at l_* inserted into both pieces.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_star_clip = float(np.clip(l_star, x[0], x[-1]))
    y_star = interp_positive_curve_logy(x, y, x_star_clip)

    left_mask = x < x_star_clip
    right_mask = x > x_star_clip

    x_dashed = np.concatenate([x[left_mask], [x_star_clip]])
    y_dashed = np.concatenate([y[left_mask], [y_star]])

    x_solid = np.concatenate([[x_star_clip], x[right_mask]])
    y_solid = np.concatenate([[y_star], y[right_mask]])

    return x_dashed, y_dashed, x_solid, y_solid, x_star_clip, y_star


def make_alpha_colors(alphas, cmap_name=BASE_CMAP_NAME):
    nA = len(alphas)
    base_cmap = plt.get_cmap(cmap_name)
    cvals = [shade_from_index(i, nA) for i in range(nA)]
    return [base_cmap(cv) for cv in cvals]

In [ ]:
# =========================
# New Cell 2 — compute derf curves, l_* points, and J_small_alpha
# =========================

# reuse the notebook parameters already defined above
sigma_21 = sigma_W1 * sigma_W2
l_arr = np.arange(L + 1, dtype=float)

colors = make_alpha_colors(alphas)

derf_lstar_curves = []

for color, alpha in zip(colors, alphas):
    chi_d, q_d, p_d = simulate_derf_chi_mlp(
        num_layers=L,
        p0=p0,
        n_tokens=n_tokens,
        alpha=float(alpha),
        sigma_W1=sigma_W1,
        sigma_W2=sigma_W2,
        sigma_O=sigma_O,
        sigma_V=sigma_V,
        sigma_A=sigma_A,
        q0=q0,
    )

    logJ_d = logJ_from_chi(chi_d)
    J_d = safe_exp(logJ_d)

    l_star = compute_l_star(alpha=float(alpha), sigma_21=sigma_21, q0=q0, p0=p0)
    J_star = interp_positive_curve_logy(l_arr, J_d, l_star)

    x_dashed, y_dashed, x_solid, y_solid, x_star_clip, y_star_clip = split_curve_at_l_star(
        l_arr, J_d, l_star
    )

    J_small_alpha = compute_J_small_alpha(
        alpha=float(alpha),
        sigma_21=sigma_21,
        l_values=l_arr,
    )

    derf_lstar_curves.append({
        "alpha": float(alpha),
        "color": color,
        "q": q_d,
        "p": p_d,
        "logJ": logJ_d,
        "J": J_d,
        "l_star": float(l_star),
        "J_star": float(J_star),
        "l_star_in_range": bool((l_arr[0] <= l_star) and (l_star <= l_arr[-1])),
        "x_dashed": x_dashed,
        "y_dashed": y_dashed,
        "x_solid": x_solid,
        "y_solid": y_solid,
        "J_small_alpha": J_small_alpha,
    })

print("alpha      l_*          J(l_*)")
for d in derf_lstar_curves:
    print(f"{d['alpha']:.2f}     {d['l_star']:.6f}     {d['J_star']:.6f}")

In [ ]:
# =========================
# New Cell 3 — plot derf curves split at l_* + grey dashed J_small_alpha
# =========================

from matplotlib.lines import Line2D

fig = plt.figure(figsize=(5.2, 5.2))
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1.0, 0.16],
    hspace=0.42 + LEGEND_VPAD
)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[1, 0])
center_shrink_axis(cax)

# grey dashed small-alpha curves
for d in derf_lstar_curves:
    ax.plot(
        l_arr,
        d["J_small_alpha"],
        color=SMALL_ALPHA_COLOR,
        linewidth=SMALL_ALPHA_LW,
        linestyle=SMALL_ALPHA_STYLE,
        dash_capstyle="round",
        zorder=1,
    )

# derf curves: dashed for l < l_*, solid for l > l_*
for d in derf_lstar_curves:
    ax.plot(
        d["x_dashed"],
        d["y_dashed"],
        color=d["color"],
        linewidth=DERF_LINEWIDTH,
        linestyle=DERF_DASHED_STYLE,
        dash_capstyle="round",
        zorder=3,
    )
    ax.plot(
        d["x_solid"],
        d["y_solid"],
        color=d["color"],
        linewidth=DERF_LINEWIDTH,
        linestyle=STYLE_SOLID,
        zorder=4,
    )

    if d["l_star_in_range"]:
        ax.scatter(
            d["l_star"],
            d["J_star"],
            s=LSTAR_MARKER_SIZE,
            color=d["color"],
            edgecolors="black",
            linewidths=0.5,
            zorder=5,
        )

ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\mathcal{J}^{\,l,0}$")
ax.set_xlim(0, L)
ax.set_title(r"derf: $\mathcal{J}^{\,l,0}$ with $l_*$ and $J_{\mathrm{small}\,\alpha}$")
prettify_axes(ax)

legend_handles = [
    Line2D([0], [0], color="black", lw=DERF_LINEWIDTH, linestyle=DERF_DASHED_STYLE,
           label=r"derf, $l<l_*$"),
    Line2D([0], [0], color="black", lw=DERF_LINEWIDTH, linestyle=STYLE_SOLID,
           label=r"derf, $l>l_*$"),
    Line2D([0], [0], color=SMALL_ALPHA_COLOR, lw=SMALL_ALPHA_LW, linestyle=SMALL_ALPHA_STYLE,
           label=r"$J_{\mathrm{small}\,\alpha}$"),
    Line2D([0], [0], marker="o", color="black", lw=0, markersize=6,
           label=r"$(l_*,\,\mathcal{J}(l_*))$"),
]
ax.legend(handles=legend_handles, frameon=False, loc="best")

add_alpha_colorbar_horizontal_single(cax, alphas, colors, label=r"$\alpha$")

plt.show()

## CIFAR direct-APJN overlays from `fit_hist_bundle`

These cells load the fit/scatter bundle once, use only `fit_hist_bundle`, and overlay raw direct APJN points from all samples against the asymptotic expressions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load/prepare once, then re-plot from `fit_scatter_plot_data`.
from pathlib import Path

try:
    prepare_fit_and_scatter_plot_data
except NameError:
    from vit_apjn_notebook_helpers import prepare_fit_and_scatter_plot_data

# fit_hist_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/fit_hist_depth64_stride8_determ_seed_124_mask_False_num_inits_8/results.pkl"
fit_hist_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/inverse_points_depth128_nlayers16_asymp_seed_124_num_inits_4_attn_mult_2.0_mlp_mult_2.0/results.pkl"
random_inverse_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"
random_direct_bundle = "/content/drive/MyDrive/ml_projects/mapes_variance/random_direct_depth32_layers5_test/results.pkl"

fit_scatter_plot_data = prepare_fit_and_scatter_plot_data(
    fit_hist_bundle,
    random_inverse_bundle,
    random_direct_bundle,
)

fit_hist_only_bundle = fit_scatter_plot_data["fit_bundle"]
# ALPHAS = [key for key in fit_hist_only_bundle['samples'][0]['direct_fit']['vit_points']['derf'].keys()]
ALPHAS = [key for key in fit_scatter_plot_data['fit_bundle']['samples'][0]['inverse_fit']['vit_points']['derf'].keys()]
TARGET_ALPHA = ALPHAS[-1]

print(f"Loaded fit bundle with {len(fit_hist_only_bundle['samples'])} samples.")
print(f"Target derf alpha = {TARGET_ALPHA}")

In [ ]:
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt


def _collect_direct_points_from_fit_bundle(fit_bundle, *, alpha):
    derf_points = []
    preln_points = []
    derf_by_layer = defaultdict(list)
    preln_by_layer = defaultdict(list)

    for sample in fit_bundle["samples"]:
        direct_fit = sample["direct_fit"]
        vit_points = direct_fit["vit_points"]

        for l, v in vit_points.get("preln", {}).items():
            l_int = int(l)
            v_float = float(v)
            preln_points.append((l_int, v_float))
            preln_by_layer[l_int].append(v_float)

        derf_dict = vit_points.get("derf", {})
        if float(alpha) in derf_dict:
            for l, v in derf_dict[float(alpha)].items():
                l_int = int(l)
                v_float = float(v)
                derf_points.append((l_int, v_float))
                derf_by_layer[l_int].append(v_float)

    if not derf_points:
        raise ValueError(f"No direct derf APJN points found for alpha={alpha}.")
    if not preln_points:
        raise ValueError("No direct pre-LN APJN points found in fit_hist_bundle.")

    return {
        "derf_points": derf_points,
        "preln_points": preln_points,
        "derf_by_layer": {int(k): np.asarray(v, dtype=float) for k, v in derf_by_layer.items()},
        "preln_by_layer": {int(k): np.asarray(v, dtype=float) for k, v in preln_by_layer.items()},
    }


def _mean_curve(layer_to_vals):
    layers = np.asarray(sorted(int(k) for k in layer_to_vals.keys()), dtype=int)
    vals = np.asarray([np.mean(layer_to_vals[int(l)]) for l in layers], dtype=float)
    return layers, vals


def _tail_mask(n, fraction=0.40):
    idx = np.arange(int(n), dtype=int)
    start = max(0, int(np.floor((1.0 - float(fraction)) * max(int(n) - 1, 0))))
    return idx >= start


def _fit_log_constant_from_points(layers, values, shape_log, fraction=0.40):
    layers = np.asarray(layers, dtype=float)
    values = np.asarray(values, dtype=float)
    shape_log = np.asarray(shape_log, dtype=float)
    valid = np.isfinite(layers) & np.isfinite(values) & np.isfinite(shape_log) & (layers > 0) & (values > 0)
    if not np.any(valid):
        raise ValueError("No valid points available for asymptotic constant fitting.")
    valid_idx = np.flatnonzero(valid)
    tail = _tail_mask(valid_idx.size, fraction=float(fraction))
    use_idx = valid_idx[tail]
    return float(np.mean(np.log(values[use_idx]) - shape_log[use_idx]))


fit_overlay_data = _collect_direct_points_from_fit_bundle(fit_hist_only_bundle, alpha=TARGET_ALPHA)
derf_layers_mean, derf_vals_mean = _mean_curve(fit_overlay_data["derf_by_layer"])
preln_layers_mean, preln_vals_mean = _mean_curve(fit_overlay_data["preln_by_layer"])

FIT_TAIL_FRACTION_OVERLAY = globals().get("FIT_FRACTION", 0.40)
sigma_21 = float(globals().get("sigma_21", sigma_W1 * sigma_W2))
sigma_OV = float(globals().get("sigma_OV", sigma_O * sigma_V * sigma_A))
zeta_current = float((((sigma_21**2) / 2.0)) / (((sigma_21**2) / 2.0) + sigma_OV**2))
if "p_star" in globals():
    p_star_current = float(p_star)
else:
    p_star_current, _, _ = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)

C_pref_target = float(TARGET_ALPHA) * 2.0 / np.pi
lam_with_pstar = ((((sigma_21**2) / 2.0) + sigma_OV**2 * p_star_current) / (C_pref_target**2 * sigma_21**4))
lam_pstar_zero = ((((sigma_21**2) / 2.0) + sigma_OV**2 * 0.0) / (C_pref_target**2 * sigma_21**4))

derf_shape_with_pstar = np.sqrt(derf_layers_mean / lam_with_pstar) - (1.0 / (8.0 * lam_with_pstar))
derf_shape_pstar_zero = np.sqrt(derf_layers_mean / lam_pstar_zero) - (1.0 / (8.0 * lam_pstar_zero))
C_derf_with_pstar = _fit_log_constant_from_points(derf_layers_mean, derf_vals_mean, derf_shape_with_pstar, fraction=FIT_TAIL_FRACTION_OVERLAY)
C_derf_pstar_zero = _fit_log_constant_from_points(derf_layers_mean, derf_vals_mean, derf_shape_pstar_zero, fraction=FIT_TAIL_FRACTION_OVERLAY)

preln_shape_current = zeta_current * np.log(preln_layers_mean)
preln_shape_zeta_one = 1.0 * np.log(preln_layers_mean)
C_preln_current = _fit_log_constant_from_points(preln_layers_mean, preln_vals_mean, preln_shape_current, fraction=FIT_TAIL_FRACTION_OVERLAY)
C_preln_zeta_one = _fit_log_constant_from_points(preln_layers_mean, preln_vals_mean, preln_shape_zeta_one, fraction=FIT_TAIL_FRACTION_OVERLAY)

print(f"Collected {len(fit_overlay_data['derf_points'])} derf direct points at alpha={TARGET_ALPHA}.")
print(f"Collected {len(fit_overlay_data['preln_points'])} pre-LN direct points.")
print(f"sigma_21 = {sigma_21:.8g}")
print(f"sigma_OV = {sigma_OV:.8g}")
print(f"p_star = {p_star_current:.8g}")
print(f"lambda(p_star) = {lam_with_pstar:.8g}")
print(f"lambda(p_star=0) = {lam_pstar_zero:.8g}")
print(f"zeta = {zeta_current:.8g}")

In [ ]:
# Derf direct APJN: raw points for all items at TARGET_ALPHA, plus two asymptotic curves.
fig = plt.figure(figsize=(5.6, 5.2))
ax = fig.add_subplot(111)

derf_points = np.asarray(fit_overlay_data["derf_points"], dtype=float)
ax.scatter(
    derf_points[:, 0],
    np.log(np.maximum(derf_points[:, 1], 1e-300)) ** 2,
    color="#1f77b4",
    alpha=0.28,
    s=18,
    label=fr"direct derf points ($\alpha={TARGET_ALPHA}$)",
    zorder=2,
)

ax.plot(
    derf_layers_mean,
    (derf_shape_with_pstar + C_derf_with_pstar) ** 2,
    color="#d62728",
    linewidth=2.3,
    label=fr"asymptotic with $p_\star$ (" + fr"$\lambda={lam_with_pstar:.3g}$)",
    zorder=4,
)
ax.plot(
    derf_layers_mean,
    (derf_shape_pstar_zero + C_derf_pstar_zero) ** 2,
    color="#2ca02c",
    linewidth=2.3,
    linestyle="--",
    label=fr"asymptotic with $p_\star=0$ (" + fr"$\lambda={lam_pstar_zero:.3g}$)",
    zorder=5,
)

ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\left(\log \mathcal{J}^{\,l,0}\right)^2$")
ax.set_title(fr"direct derf APJN at $\alpha={TARGET_ALPHA}$")
prettify_axes(ax)
ax.legend(frameon=False, loc="best")
plt.show()

In [ ]:
# Pre-LN direct APJN: raw points for all items, plus zeta and zeta=1 asymptotics.
fig = plt.figure(figsize=(5.6, 5.2))
ax = fig.add_subplot(111)

preln_points = np.asarray(fit_overlay_data["preln_points"], dtype=float)
ax.scatter(
    preln_points[:, 0],
    preln_points[:, 1],
    color="black",
    alpha=0.25,
    s=18,
    label="direct pre-LN points",
    zorder=2,
)

ax.plot(
    preln_layers_mean,
    np.exp(preln_shape_current + C_preln_current),
    color="#d62728",
    linewidth=2.3,
    label=fr"asymptotic with $\zeta={zeta_current:.3g}$",
    zorder=4,
)
ax.plot(
    preln_layers_mean,
    np.exp(preln_shape_zeta_one + C_preln_zeta_one),
    color="#2ca02c",
    linewidth=2.3,
    linestyle="--",
    label=r"asymptotic with $\zeta=1$",
    zorder=5,
)

ax.set_xscale("log")
ax.set_yscale("log")
prettify_log_axis(ax, "x")
prettify_log_axis(ax, "y")
ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\mathcal{J}^{\,l,0}$")
ax.set_title(r"direct pre-LN APJN")
prettify_axes(ax)
ax.legend(frameon=False, loc="best")
plt.show()

In [ ]:
# Inverse-APJN helpers and data extraction from fit_hist_bundle.
def _collect_inverse_points_from_fit_bundle(fit_bundle, *, alpha):
    derf_points = []
    preln_points = []
    derf_by_layer = defaultdict(list)
    preln_by_layer = defaultdict(list)

    for sample in fit_bundle["samples"]:
        inverse_fit = sample["inverse_fit"]
        vit_points = inverse_fit["vit_points"]
        layers = [int(x) + 1 for x in np.asarray(vit_points["x_shift"], dtype=int)]

        preln_vals = np.asarray(vit_points.get("preln", []), dtype=float)
        for l, v in zip(layers, preln_vals):
            v_float = float(v)
            preln_points.append((int(l), v_float))
            preln_by_layer[int(l)].append(v_float)

        derf_dict = vit_points.get("derf", {})
        if float(alpha) in derf_dict:
            derf_vals = np.asarray(derf_dict[float(alpha)], dtype=float)
            for l, v in zip(layers, derf_vals):
                v_float = float(v)
                derf_points.append((int(l), v_float))
                derf_by_layer[int(l)].append(v_float)

    if not derf_points:
        raise ValueError(f"No inverse derf APJN points found for alpha={alpha}.")
    if not preln_points:
        raise ValueError("No inverse pre-LN APJN points found in fit_hist_bundle.")

    return {
        "derf_points": derf_points,
        "preln_points": preln_points,
        "derf_by_layer": {int(k): np.asarray(v, dtype=float) for k, v in derf_by_layer.items()},
        "preln_by_layer": {int(k): np.asarray(v, dtype=float) for k, v in preln_by_layer.items()},
    }


inverse_overlay_data = _collect_inverse_points_from_fit_bundle(fit_hist_only_bundle, alpha=TARGET_ALPHA)
inverse_derf_layers_mean, inverse_derf_vals_mean = _mean_curve(inverse_overlay_data["derf_by_layer"])
inverse_preln_layers_mean, inverse_preln_vals_mean = _mean_curve(inverse_overlay_data["preln_by_layer"])

# If the stored data do not determine L unambiguously for your use case, set this by hand.
L_INVERSE_OVERRIDE = None
L_inverse = int(L_INVERSE_OVERRIDE) if L_INVERSE_OVERRIDE is not None else int(fit_hist_only_bundle.get("depth", np.max(inverse_preln_layers_mean)))
if L_inverse <= int(np.max(inverse_preln_layers_mean)):
    print("Warning: L_inverse is not larger than the largest stored inverse layer index.")


sigma_21 = float(globals().get("sigma_21", sigma_W1 * sigma_W2))
sigma_OV = float(globals().get("sigma_OV", sigma_O * sigma_V * sigma_A))
zeta_current = float((((sigma_21**2) / 2.0)) / (((sigma_21**2) / 2.0) + sigma_OV**2))
if "p_star" in globals():
    p_star_current = float(p_star)
else:
    p_star_current, _, _ = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)
C_pref_target = float(TARGET_ALPHA) * 2.0 / np.pi
lam_with_pstar = ((((sigma_21**2) / 2.0) + sigma_OV**2 * p_star_current) / (C_pref_target**2 * sigma_21**4))
lam_pstar_zero = ((((sigma_21**2) / 2.0) + sigma_OV**2 * 0.0) / (C_pref_target**2 * sigma_21**4))

inverse_curve_layers = np.arange(1, L_inverse + 1, dtype=int)
inverse_log_derf_with_pstar = (np.sqrt(float(L_inverse)) - np.sqrt(inverse_curve_layers)) / np.sqrt(lam_with_pstar)
inverse_log_derf_pstar_zero = (np.sqrt(float(L_inverse)) - np.sqrt(inverse_curve_layers)) / np.sqrt(lam_pstar_zero)
inverse_preln_curve_current = (float(L_inverse) / inverse_curve_layers) ** zeta_current
inverse_preln_curve_zeta_one = (float(L_inverse) / inverse_curve_layers) ** 1.0

print(f"Collected {len(inverse_overlay_data['derf_points'])} inverse derf points at alpha={TARGET_ALPHA}.")
print(f"Collected {len(inverse_overlay_data['preln_points'])} inverse pre-LN points.")
print(f"Using L_inverse = {L_inverse}")

In [ ]:
# Inverse derf APJN: raw points for all items at TARGET_ALPHA, plus two asymptotic curves.
fig = plt.figure(figsize=(5.6, 5.2))
ax = fig.add_subplot(111)

inverse_derf_points = np.asarray(inverse_overlay_data["derf_points"], dtype=float)
ax.scatter(
    inverse_derf_points[:, 0],
    np.log(np.maximum(inverse_derf_points[:, 1], 1e-300)),
    color="#1f77b4",
    alpha=0.28,
    s=18,
    label=fr"inverse derf points ($\alpha={TARGET_ALPHA}$)",
    zorder=2,
)

ax.plot(
    inverse_curve_layers,
    inverse_log_derf_with_pstar,
    color="#d62728",
    linewidth=2.3,
    label=fr"$(\sqrt{{L}}-\sqrt{{l}})/\sqrt{{\lambda}}$, $p_\star$" + fr" ($\lambda={lam_with_pstar:.3g}$)",
    zorder=4,
)
ax.plot(
    inverse_curve_layers,
    inverse_log_derf_pstar_zero,
    color="#2ca02c",
    linewidth=2.3,
    linestyle="--",
    label=fr"$(\sqrt{{L}}-\sqrt{{l}})/\sqrt{{\lambda}}$, $p_\star=0$" + fr" ($\lambda={lam_pstar_zero:.3g}$)",
    zorder=5,
)

ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\log \mathcal{J}^{\,L,l}$")
ax.set_title(fr"inverse derf APJN at $\alpha={TARGET_ALPHA}$")
prettify_axes(ax)
ax.legend(frameon=False, loc="best")
plt.show()

In [ ]:
# Inverse pre-LN APJN: raw points for all items, plus zeta and zeta=1 curves.
fig = plt.figure(figsize=(5.6, 5.2))
ax = fig.add_subplot(111)

inverse_preln_points = np.asarray(inverse_overlay_data["preln_points"], dtype=float)
ax.scatter(
    inverse_preln_points[:, 0],
    inverse_preln_points[:, 1],
    color="black",
    alpha=0.25,
    s=18,
    label="inverse pre-LN points",
    zorder=2,
)

ax.plot(
    inverse_curve_layers,
    inverse_preln_curve_current,
    color="#d62728",
    linewidth=2.3,
    label=fr"$(L/l)^\zeta$ with $\zeta={zeta_current:.3g}$",
    zorder=4,
)
ax.plot(
    inverse_curve_layers,
    inverse_preln_curve_zeta_one,
    color="#2ca02c",
    linewidth=2.3,
    linestyle="--",
    label=r"$(L/l)^{\zeta}$ with $\zeta=1$",
    zorder=5,
)

ax.set_xscale("log")
ax.set_yscale("log")
prettify_log_axis(ax, "x")
prettify_log_axis(ax, "y")
ax.set_xlabel(r"$l$")
ax.set_ylabel(r"$\mathcal{J}^{\,L,l}$")
ax.set_title(r"inverse pre-LN APJN")
prettify_axes(ax)
ax.legend(frameon=False, loc="best")
plt.show()

**MORE POINTS**

In [ ]:
# Colab/local setup for vit-dyt.
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('vit-dyt')
INSTALL_TIMM = True
INSTALL_REQUIREMENTS = True
CLONE_IF_MISSING = True

if INSTALL_TIMM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

if not REPO_DIR.exists() and CLONE_IF_MISSING:
    subprocess.run(['git', 'clone', 'https://github.com/labofdoubt/vit-dyt', str(REPO_DIR)], check=True)

REQ = REPO_DIR / 'requirements.txt'
if INSTALL_REQUIREMENTS and REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)

if REPO_DIR.exists() and str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

print('Repo directory:', REPO_DIR.resolve() if REPO_DIR.exists() else 'missing')

In [ ]:
from vit_apjn_notebook_helpers import *

bootstrap_vit_dyt_repo(
    REPO_DIR,
    clone_if_missing=CLONE_IF_MISSING,
    install_requirements=INSTALL_REQUIREMENTS,
    install_timm=INSTALL_TIMM,
)
print('DEVICE =', DEVICE)

In [ ]:
MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=128,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)

In [ ]:
# Inverse-only CIFAR APJN sampling with random internal layers and optional attention-value scaling.
DEFAULT_ALPHAS = (1.0,)
INVERSE_ONLY_ATTN_MULT = 1.0
INVERSE_ONLY_MLP_MULT = 1.0

INVERSE_ONLY_N_SAMPLES = 80
INVERSE_ONLY_NUM_LAYERS = 16
INVERSE_ONLY_BATCH_SEED = 124
INVERSE_ONLY_STD_THRESHOLD = 0.0
INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH = 20
INVERSE_ONLY_J_NUM_DRAWS = 10
INVERSE_ONLY_NUM_MODEL_INITS = 4
INVERSE_ONLY_DETERMINISTIC = True
INVERSE_ONLY_SAVE_RESULTS = True
INVERSE_ONLY_REWRITE = False
INVERSE_ONLY_SAVE_EVERY = 8

INVERSE_ONLY_RESULTS_POSTFIX = f'asymp_seed_{INVERSE_ONLY_BATCH_SEED}_num_inits_{INVERSE_ONLY_NUM_MODEL_INITS}_attn_mult_{INVERSE_ONLY_ATTN_MULT}_mlp_mult_{INVERSE_ONLY_MLP_MULT}'

inverse_only_bundle = run_cifar_inverse_apjn_points(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    n_samples=INVERSE_ONLY_N_SAMPLES,
    num_layers=INVERSE_ONLY_NUM_LAYERS,
    batch_seed=INVERSE_ONLY_BATCH_SEED,
    std_threshold=INVERSE_ONLY_STD_THRESHOLD,
    max_epochs_to_search=INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH,
    j_num_draws=INVERSE_ONLY_J_NUM_DRAWS,
    num_model_inits=INVERSE_ONLY_NUM_MODEL_INITS,
    deterministic=INVERSE_ONLY_DETERMINISTIC,
    attn_mult=INVERSE_ONLY_ATTN_MULT,
    mlp_mult=INVERSE_ONLY_MLP_MULT,
    save_every_n_samples=INVERSE_ONLY_SAVE_EVERY,
    save_results=INVERSE_ONLY_SAVE_RESULTS,
    rewrite=INVERSE_ONLY_REWRITE,
    result_postfix=INVERSE_ONLY_RESULTS_POSTFIX,
)



In [ ]:
# Inverse-only CIFAR APJN sampling with random internal layers and optional attention-value scaling.
DEFAULT_ALPHAS = (1.0,)
INVERSE_ONLY_ATTN_MULT = 2.0
INVERSE_ONLY_MLP_MULT = 1.0

INVERSE_ONLY_N_SAMPLES = 80
INVERSE_ONLY_NUM_LAYERS = 16
INVERSE_ONLY_BATCH_SEED = 124
INVERSE_ONLY_STD_THRESHOLD = 0.0
INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH = 20
INVERSE_ONLY_J_NUM_DRAWS = 10
INVERSE_ONLY_NUM_MODEL_INITS = 4
INVERSE_ONLY_DETERMINISTIC = True
INVERSE_ONLY_SAVE_RESULTS = True
INVERSE_ONLY_REWRITE = False
INVERSE_ONLY_SAVE_EVERY = 8

INVERSE_ONLY_RESULTS_POSTFIX = f'asymp_seed_{INVERSE_ONLY_BATCH_SEED}_num_inits_{INVERSE_ONLY_NUM_MODEL_INITS}_attn_mult_{INVERSE_ONLY_ATTN_MULT}_mlp_mult_{INVERSE_ONLY_MLP_MULT}'

inverse_only_bundle = run_cifar_inverse_apjn_points(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    n_samples=INVERSE_ONLY_N_SAMPLES,
    num_layers=INVERSE_ONLY_NUM_LAYERS,
    batch_seed=INVERSE_ONLY_BATCH_SEED,
    std_threshold=INVERSE_ONLY_STD_THRESHOLD,
    max_epochs_to_search=INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH,
    j_num_draws=INVERSE_ONLY_J_NUM_DRAWS,
    num_model_inits=INVERSE_ONLY_NUM_MODEL_INITS,
    deterministic=INVERSE_ONLY_DETERMINISTIC,
    attn_mult=INVERSE_ONLY_ATTN_MULT,
    mlp_mult=INVERSE_ONLY_MLP_MULT,
    save_every_n_samples=INVERSE_ONLY_SAVE_EVERY,
    save_results=INVERSE_ONLY_SAVE_RESULTS,
    rewrite=INVERSE_ONLY_REWRITE,
    result_postfix=INVERSE_ONLY_RESULTS_POSTFIX,
)



In [ ]:
# Inverse-only CIFAR APJN sampling with random internal layers and optional attention-value scaling.
DEFAULT_ALPHAS = (1.0,)
INVERSE_ONLY_ATTN_MULT = 1.0
INVERSE_ONLY_MLP_MULT = 2.0

INVERSE_ONLY_N_SAMPLES = 80
INVERSE_ONLY_NUM_LAYERS = 16
INVERSE_ONLY_BATCH_SEED = 124
INVERSE_ONLY_STD_THRESHOLD = 0.0
INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH = 20
INVERSE_ONLY_J_NUM_DRAWS = 10
INVERSE_ONLY_NUM_MODEL_INITS = 4
INVERSE_ONLY_DETERMINISTIC = True
INVERSE_ONLY_SAVE_RESULTS = True
INVERSE_ONLY_REWRITE = False
INVERSE_ONLY_SAVE_EVERY = 8

INVERSE_ONLY_RESULTS_POSTFIX = f'asymp_seed_{INVERSE_ONLY_BATCH_SEED}_num_inits_{INVERSE_ONLY_NUM_MODEL_INITS}_attn_mult_{INVERSE_ONLY_ATTN_MULT}_mlp_mult_{INVERSE_ONLY_MLP_MULT}'

inverse_only_bundle = run_cifar_inverse_apjn_points(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    n_samples=INVERSE_ONLY_N_SAMPLES,
    num_layers=INVERSE_ONLY_NUM_LAYERS,
    batch_seed=INVERSE_ONLY_BATCH_SEED,
    std_threshold=INVERSE_ONLY_STD_THRESHOLD,
    max_epochs_to_search=INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH,
    j_num_draws=INVERSE_ONLY_J_NUM_DRAWS,
    num_model_inits=INVERSE_ONLY_NUM_MODEL_INITS,
    deterministic=INVERSE_ONLY_DETERMINISTIC,
    attn_mult=INVERSE_ONLY_ATTN_MULT,
    mlp_mult=INVERSE_ONLY_MLP_MULT,
    save_every_n_samples=INVERSE_ONLY_SAVE_EVERY,
    save_results=INVERSE_ONLY_SAVE_RESULTS,
    rewrite=INVERSE_ONLY_REWRITE,
    result_postfix=INVERSE_ONLY_RESULTS_POSTFIX,
)



In [ ]:
# Inverse-only CIFAR APJN sampling with random internal layers and optional attention-value scaling.
DEFAULT_ALPHAS = (1.0,)
INVERSE_ONLY_ATTN_MULT = 2.0
INVERSE_ONLY_MLP_MULT = 2.0

INVERSE_ONLY_N_SAMPLES = 80
INVERSE_ONLY_NUM_LAYERS = 16
INVERSE_ONLY_BATCH_SEED = 124
INVERSE_ONLY_STD_THRESHOLD = 0.0
INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH = 20
INVERSE_ONLY_J_NUM_DRAWS = 10
INVERSE_ONLY_NUM_MODEL_INITS = 4
INVERSE_ONLY_DETERMINISTIC = True
INVERSE_ONLY_SAVE_RESULTS = True
INVERSE_ONLY_REWRITE = False
INVERSE_ONLY_SAVE_EVERY = 8

INVERSE_ONLY_RESULTS_POSTFIX = f'asymp_seed_{INVERSE_ONLY_BATCH_SEED}_num_inits_{INVERSE_ONLY_NUM_MODEL_INITS}_attn_mult_{INVERSE_ONLY_ATTN_MULT}_mlp_mult_{INVERSE_ONLY_MLP_MULT}'

inverse_only_bundle = run_cifar_inverse_apjn_points(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    n_samples=INVERSE_ONLY_N_SAMPLES,
    num_layers=INVERSE_ONLY_NUM_LAYERS,
    batch_seed=INVERSE_ONLY_BATCH_SEED,
    std_threshold=INVERSE_ONLY_STD_THRESHOLD,
    max_epochs_to_search=INVERSE_ONLY_MAX_EPOCHS_TO_SEARCH,
    j_num_draws=INVERSE_ONLY_J_NUM_DRAWS,
    num_model_inits=INVERSE_ONLY_NUM_MODEL_INITS,
    deterministic=INVERSE_ONLY_DETERMINISTIC,
    attn_mult=INVERSE_ONLY_ATTN_MULT,
    mlp_mult=INVERSE_ONLY_MLP_MULT,
    save_every_n_samples=INVERSE_ONLY_SAVE_EVERY,
    save_results=INVERSE_ONLY_SAVE_RESULTS,
    rewrite=INVERSE_ONLY_REWRITE,
    result_postfix=INVERSE_ONLY_RESULTS_POSTFIX,
)